# Capstone Project: Comparative Analysis of Traditional and AI-Driven Threat Detection
## Course: AI in Cybersecurity | B.Tech Final Year Evaluation
**System:** AI-Based Web Application Firewall (AI-WAF)

**Core Models:** PyTorch 1D-CNN (Sequence Deep Learning) + XGBoost (Domain Features) + Traditional Rule Engine

**Corpus:** Spanish National Research Council (CSIC 2010) HTTP Dataset (25,608 unique records, 0% leakage)

---

In [1]:
import sys, os, pathlib, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import joblib
from scipy import sparse

# Add workspace root to Python path
ROOT_DIR = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print(f'Workspace Root: {ROOT_DIR}')
print(f'PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')

Workspace Root: C:\Users\Acer\Downloads\AI-BASED WEB APPLICATION FIREWALL
PyTorch Version: 2.14.1+cpu | CUDA Available: False


## Activity 2 & 3: Dataset Ingestion, Deduplication Audit & EDA
We examine the raw CSIC 2010 dataset and verify the critical session-cookie deduplication.

In [2]:
split_meta_path = ROOT_DIR / 'data' / 'processed' / 'split_metadata.json'
with open(split_meta_path, 'r') as f:
    split_meta = json.load(f)

print('--- Corpus Partition & Leakage Audit ---')
print(json.dumps(split_meta, indent=2))

test_df = pd.read_csv(ROOT_DIR / 'data' / 'processed' / 'test_frozen.csv')
print(f'\nFrozen Test Set Shape: {test_df.shape}')
print(test_df['diagnostic_category'].value_counts())

--- Corpus Partition & Leakage Audit ---
{
  "random_seed": 42,
  "split_ratio": {
    "train": 0.7,
    "validation": 0.15,
    "test": 0.15
  },
  "row_counts": {
    "total_deduplicated": 25608,
    "train": 17925,
    "validation": 3841,
    "test_frozen": 3842
  },
  "class_distributions": {
    "train": {
      "1": 11174,
      "0": 6751
    },
    "validation": {
      "1": 2395,
      "0": 1446
    },
    "test_frozen": {
      "1": 2395,
      "0": 1447
    }
  },
  "class_balance_percentages": {
    "train_anomalous_pct": 62.34,
    "validation_anomalous_pct": 62.35,
    "test_anomalous_pct": 62.34
  },
  "leakage_audit": {
    "train_val_overlap": 0,
    "train_test_overlap": 0,
    "val_test_overlap": 0
  }
}

Frozen Test Set Shape: (3842, 14)
diagnostic_category
GENERIC_ANOMALY    1898
BENIGN             1447
SQL_INJECTION       246
XSS                 144
CRLF_INJECTION      107
Name: count, dtype: int64


## Activity 4: Feature Engineering & Selection Analysis
We inspect the 79 handcrafted numerical features and the feature selection results from Random Forest, ANOVA F-statistic, and RFE.

In [3]:
feat_meta_path = ROOT_DIR / 'data' / 'processed' / 'feature_metadata.json'
with open(feat_meta_path, 'r') as f:
    feat_meta = json.load(f)

dims = feat_meta['feature_dimensions']
print(f"Total Handcrafted Features: {dims['handcrafted_numerical_features']}")
print(f"TF-IDF Character n-grams: {dims['tfidf_char_ngram_features']}")
print(f"Total Combined Representation: {dims['total_combined_features']}")
print('\nTop 10 Most Influential Features (Random Forest Gini Importance):')
for item in feat_meta['top_10_features_by_rf_importance']:
    print(f"  - {item['feature']:<30}: {item['rf_importance']:.4f}")

Total Handcrafted Features: 79
TF-IDF Character n-grams: 500
Total Combined Representation: 579

Top 10 Most Influential Features (Random Forest Gini Importance):
  - ratio_uppers                  : 0.1202
  - shannon_entropy               : 0.0805
  - ratio_alphas                  : 0.0730
  - ratio_specials                : 0.0679
  - ratio_digits                  : 0.0643
  - ratio_slash                   : 0.0594
  - avg_param_length              : 0.0540
  - ratio_space                   : 0.0509
  - ratio_equal                   : 0.0497
  - count_slash                   : 0.0449


## Activity 5: Machine Learning Models & Master Benchmark
We evaluate all trained models on the untouched frozen test set (3,842 requests).

In [4]:
benchmark_csv = ROOT_DIR / 'reports' / 'final_model_benchmark_test_set.csv'
df_bench = pd.read_csv(benchmark_csv)
display = df_bench[['System', 'Accuracy', 'Precision', 'Recall', 'F1_Score', 'ROC_AUC', 'False_Positive_Rate', 'Latency_ms']].copy()
print(display.to_string(index=False))

              System  Accuracy  Precision  Recall  F1_Score  ROC_AUC  False_Positive_Rate  Latency_ms
      PyTorch 1D CNN    0.9435     0.9986  0.9106    0.9526   0.9870               0.0021       0.866
             Xgboost    0.9094     0.9642  0.8877    0.9243   0.9740               0.0546       0.851
       Random Forest    0.8896     0.9612  0.8576    0.9064   0.9598               0.0574      37.740
 Logistic Regression    0.8251     0.8817  0.8309    0.8555   0.9073               0.1845       0.530
         PyTorch MLP    0.8186     0.8564  0.8518    0.8541   0.9203               0.2364       0.146
          Linear Svm    0.7819     0.8192  0.8342    0.8266   0.8718               0.3048       0.410
      PyTorch BiLSTM    0.6869     0.7998  0.6639    0.7255   0.7766               0.2751       3.384
 PyTorch Autoencoder    0.6936     0.9849  0.5165    0.6776   0.8453               0.0131       0.198
Traditional Rule WAF    0.5133     1.0000  0.2192    0.3596   0.6096              

## Activity 6 & 8: Ablation Study & Attack Subgroup Analysis
Evidence-based comparison demonstrating the superiority of the complete Hybrid Ensemble over isolated models.

In [5]:
with open(ROOT_DIR / 'reports' / 'ablation_study_metrics.json', 'r') as f:
    abl_data = json.load(f)

df_abl = pd.DataFrame(abl_data).T
print('=== ABLATION STUDY COMPARISON ===')
print(df_abl[['accuracy', 'precision', 'recall', 'f1_score', 'false_positive_rate', 'latency_ms']])

with open(ROOT_DIR / 'reports' / 'class_specific_metrics.json', 'r') as f:
    class_data = json.load(f)

print('\n=== ATTACK SUBGROUP DETECTION RECALL ===')
for threat, stats in class_data.items():
    if threat == 'BENIGN':
        print(f"{threat:<18} (N={stats['sample_count']}): Clean Pass Rate = {stats['ensemble_clean_allow_rate']*100:.2f}%")
    else:
        print(f"{threat:<18} (N={stats['sample_count']}): Recall = {stats['ensemble_recall']*100:.2f}% (Rules: {stats['rule_recall']*100:.2f}%, XGB: {stats['xgb_recall']*100:.2f}%, CNN: {stats['cnn_recall']*100:.2f}%)")

=== ABLATION STUDY COMPARISON ===
                     accuracy precision  recall f1_score false_positive_rate  \
Rule_Engine_Only       0.5133       1.0  0.2192   0.3596                 0.0   
XGBoost_Only           0.9094    0.9642  0.8877   0.9243              0.0546   
PyTorch_1D_CNN_Only    0.9435    0.9986  0.9106   0.9526              0.0021   
Rules_Plus_XGBoost     0.8584    0.9941  0.7775   0.8725              0.0076   
XGBoost_Plus_CNN       0.9485    0.9973  0.9198    0.957              0.0041   
Full_Hybrid_Ensemble   0.9513    0.9973  0.9244   0.9595              0.0041   

                     latency_ms  
Rule_Engine_Only          0.051  
XGBoost_Only              0.963  
PyTorch_1D_CNN_Only       0.596  
Rules_Plus_XGBoost        1.049  
XGBoost_Plus_CNN          1.586  
Full_Hybrid_Ensemble      7.145  

=== ATTACK SUBGROUP DETECTION RECALL ===
SQL_INJECTION      (N=246): Recall = 100.00% (Rules: 91.87%, XGB: 100.00%, CNN: 96.34%)
XSS                (N=144): Recall = 

## Activity 7: Explainable AI (TreeSHAP & Token Attribution)
We test the WAF decision engine on a live attack payload and inspect the AI evidence.

In [6]:
from backend.decision_engine import WAFDecisionEngine

engine = WAFDecisionEngine()
sample_attack = "/tienda1/publico/anadir.jsp?id=1%27%20UNION%20SELECT%20username,%20password%20FROM%20usuarios--"
inspection_result = engine.inspect_request(method='GET', url=sample_attack)

print(f'Target Request: {sample_attack}')
print(f"Verdict:       {inspection_result['verdict']} (HTTP Status: {inspection_result['http_status']})")
print(f"Risk Score:    {inspection_result['risk_score']} (Confidence: {inspection_result['confidence']})")
print(f"Reason:        {inspection_result['reason']}")
print(f"Rule Evidence: {inspection_result['rule_evidence']['matched_rules']}")
print(f"AI Evidence:   {inspection_result['ai_evidence']}")

[TraditionalRuleEngine] Loaded and compiled 18 security rules from C:\Users\Acer\Downloads\AI-BASED WEB APPLICATION FIREWALL\config\rules.json


Target Request: /tienda1/publico/anadir.jsp?id=1%27%20UNION%20SELECT%20username,%20password%20FROM%20usuarios--
Verdict:       BLOCK (HTTP Status: 403)
Risk Score:    1.0 (Confidence: 1.0)
Reason:        High-confidence signature triggered: SQL Injection - UNION SELECT Attack
Rule Evidence: [{'id': 'SQLI_01_UNION_SELECT', 'name': 'SQL Injection - UNION SELECT Attack', 'category': 'SQL_INJECTION', 'weight': 0.95, 'matched_substring': 'UNION SELECT'}, {'id': 'SQLI_03_COMMENTS', 'name': 'SQL Injection - Comment Syntax', 'category': 'SQL_INJECTION', 'weight': 0.7, 'matched_substring': '--'}]
AI Evidence:   {'primary_model': 'PyTorch_1D_CNN', 'attack_probability': 1.0, 'auxiliary_model': 'XGBoost', 'auxiliary_probability': 0.9999}


## Summary & Viva Defense Takeaways
1. **Traditional Rules Alone Fail:** 78.08% False Negative Rate on the CSIC 2010 test set.
2. **PyTorch 1D-CNN Champion:** 94.35% accuracy, 99.86% precision, and sub-millisecond inference (0.866 ms).
3. **Hybrid Ensemble:** Achieves 95.13% accuracy, 100% recall on SQLi/XSS, and 0.41% false alarm rate with calibrated 3-way decision logic (ALLOW, REVIEW, BLOCK).
4. **Zero-Trust Network Isolation:** Protected target application operates with zero host ports exposed on strict Docker internal network.